# Notebook 6: GenAI Assistant with RAG

The earlier notebooks produced a lot of results: category statistics, two classifiers, SHAP for both, recurring themes, cross-category topics and anomalies.
A support manager shouldn't have to dig through tables to find an answer. So we build an assistant they can ask in plain English,
for example *"Which categories have the most unhappy customers?"* or *"How much does the BGE model reduce manual review?"*

**Why RAG (Retrieval-Augmented Generation)?** A language model on its own doesn't know our data and may confidently make up numbers.
With RAG, we first **retrieve** the most relevant facts from our own results, and then the model **generates** an answer using only those facts.

```
question -> hybrid search over the knowledge base -> top 5 documents -> LLM writes the answer from them -> check the numbers
```

**Design choices**
- **Knowledge base** (about 150 documents), built automatically, with no hand-typed numbers:
  - *data documents* generated from every saved result file (notebooks 01–05 and 02b, 03b, 04b)
  - *narrative documents*: the written interpretation of every notebook section, the evaluation report and the README
- **Retrieval**: hybrid search. TF-IDF keyword similarity is combined with **BGE sentence-embedding** similarity (semantic search), so a question
  finds the right document even when it uses different words. Without `sentence-transformers` it falls back to keywords only.
- **Generation**: **Mistral** (key read from the `.env` file), Claude or any OpenAI-compatible model. If one Mistral model is rate-limited, the next one is tried.
  **Without a key it still works**: an offline mode answers with the most relevant document text.
- **Chat**: follow-up questions are understood, because the previous question is also used for retrieval and the last turns are sent to the LLM.
- **Safety**:
  - the prompt tells the model to use only the given documents
  - sensitive patterns (emails, phone numbers, IP addresses) are masked before anything is sent
  - every number in the answer is checked against the retrieved documents
- The assistant only explains. It never closes or changes tickets, which the brief puts out of scope.

The code lives in `src/rag.py`, so the dashboard uses exactly the same assistant.

In [1]:
import os
os.environ.update(TQDM_DISABLE="1", TRANSFORMERS_VERBOSITY="error", HF_HUB_DISABLE_PROGRESS_BARS="1")
import sys
import logging
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")
logging.disable(logging.WARNING)
import pandas as pd

sys.path.append("..")
from src import config, rag

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)

print("LLM provider:", rag.llm_provider())
if rag.llm_provider() == "mistral":
    print("Mistral models tried in order:", config.MISTRAL_MODELS)
else:
    print("(put a Mistral key in .env, or set ANTHROPIC_API_KEY / OPENAI_API_KEY, to use a real LLM)")

LLM provider: mistral
Mistral models tried in order: ['ministral-14b-latest', 'mistral-small-latest', 'ministral-8b-latest']


## 1. Build the knowledge base

In [2]:
kb = rag.build_knowledge_base()
print("Documents:", len(kb))
print(kb["source"].value_counts().to_string())
kb[["doc_id", "source", "title"]].sample(12, random_state=config.SEED)

Documents: 147
source
notebook     60
theme        34
model        11
report       10
category      8
shap          8
readme        7
analytics     5
anomaly       3
data          1


,doc_id,source,title
125,notebook:06#1,notebook,Notebook 6: GenAI Assistant with RAG: 1. Build the knowledge base
51,theme:other#0,theme,"Recurring issue in other: general, general question, question, question account"
138,report#8,report,Evaluation report: 9. Recurring issues and anomalies (summary)
19,themes:how_to,theme,Recurring issues in how_to (clustering)
104,notebook:04#0,notebook,Notebook 4: Recurring Issue Detection (Unsupervised Clustering): Introduction
12,shap:bug,shap,Which words drive bug predictions (SHAP explainability)
76,notebook:01#6,notebook,"Notebook 1: Sampling, Data Cleaning and EDA: 7. Correlation analysis"
31,model:leakage,model,Data leakage and train/test split
81,notebook:02#2,notebook,Notebook 2: Ticket Classification (Supervised Machine Learning): 2. Train / test split by message
9,shap:billing_problem,shap,Which words drive billing_problem predictions (SHAP explainability)


In [3]:
# what one document looks like
print(kb.loc[kb["doc_id"] == "analytics:satisfaction", "text"].iloc[0])

Customer sentiment and satisfaction (CSAT) differ a lot by category. Share of negative or very negative (unhappy, frustrated) customers and average CSAT per category, from worst to best: performance: 60.8% negative, CSAT 2.80; security_concern: 60.1% negative, CSAT 2.85; billing_problem: 59.1% negative, CSAT 2.82; account_access: 58.1% negative, CSAT 2.85; bug: 30.7% negative, CSAT 3.40; other: 29.0% negative, CSAT 3.45; feature_request: 20.0% negative, CSAT 3.73; how_to: 18.9% negative, CSAT 3.76. Problem categories make customers much unhappier than requests such as how_to and feature_request.


Data documents are generated from the saved result files, and narrative documents from the notebooks and reports.
So if a notebook is re-run with different data, the knowledge base updates automatically.

## 2. Retrieval: does the search find the right document?

Before trusting the answers, we test the search on its own. We wrote **28 questions**, including questions about the embedding models,
SHAP for the BGE model, cross-category topics, the README and the EDA. For each question we list the document (or documents) that answer it.
Some questions are phrased in everyday words on purpose (*"My customers keep getting locked out, what does the data say?"*).

We measure:
- **Hit@1**: how often a right document is the top result
- **Hit@5**: how often one is in the top 5 (the documents passed to the LLM)
- **MRR** (mean reciprocal rank): 1 if always first, 0.5 if always second, and so on

We compare keyword search alone with the hybrid search.

In [4]:
TESTS = {
    "Which categories have the most unhappy customers?": ["analytics:satisfaction"],
    "How good is the model?": ["model:summary", "model:embeddings"],
    "Is there data leakage?": ["model:leakage", "notebook:01#7", "report#1"],
    "How long do tickets take to resolve?": ["analytics:resolution"],
    "Were there any spikes in ticket volume?": ["anomaly:volume"],
    "What are the recurring bug issues?": ["themes:bug"],
    "Which words drive billing_problem predictions?": ["shap:billing_problem"],
    "Why was Logistic Regression chosen over SVM?": ["model:comparison"],
    "Which queue gets the most tickets?": ["analytics:routing"],
    "Which tickets were handled unusually?": ["anomaly:tickets"],
    "Which groups need operational attention?": ["anomaly:hotspots"],
    "Is the data synthetic?": ["data:quality"],
    "Explain the performance category": ["category:performance"],
    "Which embedding model works best?": ["model:embeddings", "notebook:02b#4", "notebook:02b#6"],
    "How much does the BGE model reduce manual review work?": ["model:confidence_rule", "notebook:02b#6"],
    "Which sentences does the best model still get wrong?": ["model:errors", "notebook:02b#5"],
    "Can we trust the SHAP explanations of the embedding model?": ["model:shap_bge", "notebook:03b#4"],
    "Are there topics that span several categories?": ["topics:overview", "notebook:04b#3"],
    "How did the recurring themes change with BGE clustering?": ["cluster:bge_vs_lsa", "notebook:04b#4", "notebook:04b#7"],
    "What F1 does account_access get with each embedding?": ["model:embedding_per_class"],
    "Do priority or channel help predict the category?": ["analytics:profile"],
    "Does the SLA plan change how fast tickets are resolved?": ["analytics:resolution"],
    "Why does Random Forest do badly on embeddings?": ["model:embeddings", "notebook:02b#4"],
    "How do I run the project?": ["readme#2"],
    "What are the limitations of this project?": ["report#9", "readme#6"],
    "What did the correlation analysis show?": ["notebook:01#6"],
    "Why do customers get frustrated with login problems?": ["category:account_access", "analytics:satisfaction"],
    "My customers keep getting locked out, what does the data say?": ["category:account_access", "themes:account_access"],
}

def evaluate(retriever):
    rows = []
    for question, ok in TESTS.items():
        found = list(retriever.search(question, k=5)["doc_id"])
        rank = next((i + 1 for i, d in enumerate(found) if d in ok), None)
        rows.append({"question": question, "expected": " / ".join(ok), "top_result": found[0], "rank": rank})
    return pd.DataFrame(rows)

def summary(r):
    return {"Hit@1": (r["rank"] == 1).mean(), "Hit@5": r["rank"].notna().mean(),
            "MRR": r["rank"].apply(lambda x: 1 / x if x else 0).mean()}

retriever = rag.Retriever(kb)                                    # hybrid if sentence-transformers is installed
keyword_only = rag.Retriever(kb, encoder=None)
retrieval = evaluate(retriever)
print("Retrieval mode:", retriever.mode)
display(pd.DataFrame({"keywords only": summary(evaluate(keyword_only)), retriever.mode: summary(retrieval)}).T
        .style.format({"Hit@1": "{:.0%}", "Hit@5": "{:.0%}", "MRR": "{:.2f}"}))
retrieval

Retrieval mode: hybrid (keywords + BGE semantic search)


,Hit@1,Hit@5,MRR
keywords only,61%,86%,0.84
hybrid (keywords + BGE semantic search),79%,96%,0.89


,question,expected,top_result,rank
0,Which categories have the most unhappy customers?,analytics:satisfaction,analytics:satisfaction,1.0
1,How good is the model?,model:summary / model:embeddings,model:summary,1.0
2,Is there data leakage?,model:leakage / notebook:01#7 / report#1,notebook:01#7,1.0
3,How long do tickets take to resolve?,analytics:resolution,analytics:resolution,1.0
4,Were there any spikes in ticket volume?,anomaly:volume,anomaly:volume,1.0
5,What are the recurring bug issues?,themes:bug,theme:bug#2,2.0
6,Which words drive billing_problem predictions?,shap:billing_problem,shap:billing_problem,1.0
7,Why was Logistic Regression chosen over SVM?,model:comparison,model:comparison,1.0
8,Which queue gets the most tickets?,analytics:routing,analytics:routing,1.0
9,Which tickets were handled unusually?,anomaly:tickets,notebook:05#3,4.0


Adding semantic search helps a lot. Hit@1 goes from 61% (keywords only) to **79%**, Hit@5 from 86% to **96%**, and MRR from 0.85 to 0.90.
The biggest gains are on everyday phrasing such as *"My customers keep getting locked out"*, which shares few words with the documents but means the same thing.

Most of the remaining rank-1 misses return a closely related document. For example, *"What are the recurring bug issues?"* returns one specific bug theme first and the full bug theme summary second.
The LLM sees all 5 documents, so the answer is still complete.

In [5]:
questions = [
    "Which categories have the most unhappy customers?",
    "Which model is best and how much better is it than TF-IDF?",
    "How much does the BGE model reduce manual review work?",
    "Are there recurring issues that span several categories?",
    "Can we trust the SHAP explanations of the embedding model?",
]
for q in questions:
    result = rag.answer(q, retriever)
    print("Q:", q)
    print("A:", result["answer"])
    print(f"   mode: {result['mode']} | sources: {result['sources']} | numbers grounded: {result['numbers_grounded']}")
    print("-" * 120)

Q: Which categories have the most unhappy customers?
A: The categories with the most unhappy customers (highest share of negative or very negative sentiment) are:

- **Performance**: 60.8% negative, CSAT 2.80
- **Security concern**: 60.1% negative, CSAT 2.85
- **Billing problem**: 59.1% negative, CSAT 2.82
- **Account access**: 58.1% negative, CSAT 2.85

These categories consistently show ~60% negative sentiment, significantly higher than the overall average of 42% [notebook:05#2, analytics:satisfaction]. The **BGE embeddings + Logistic Regression** model is the best-performing classifier for these categories, but **TF-IDF + Logistic Regression** is also referenced in the context [category:billing_problem].
   mode: LLM (mistral: ministral-14b-latest) | sources: ['analytics:satisfaction', 'notebook:05#4', 'notebook:05#2', 'notebook:01#5', 'category:billing_problem'] | numbers grounded: True
------------------------------------------------------------------------------------------------

Q: Which model is best and how much better is it than TF-IDF?
A: The **best model** is **BGE sentence embeddings + Logistic Regression** [model:embeddings].

- **Macro F1**: 0.816 (vs. 0.541 for TF-IDF + Logistic Regression).
- **Accuracy**: 0.833 (vs. 0.585 for TF-IDF).
- **Weighted F1**: 0.817 (vs. 0.541 for TF-IDF).
- **Macro ROC-AUC**: 0.994 (vs. 0.862 for TF-IDF).

This represents a **significant improvement** over the original TF-IDF + Logistic Regression model, particularly for underrepresented categories like *account_access* and *other* [model:embedding_per_class]. The BGE model also generalizes better to new wording, unlike TF-IDF, which struggles with rare or novel phrasing [model:embeddings].
   mode: LLM (mistral: ministral-14b-latest) | sources: ['model:embeddings', 'model:summary', 'model:embedding_per_class', 'notebook:03b#0', 'notebook:02b#0'] | numbers grounded: True
------------------------------------------------------------------------------------------------------

Q: How much does the BGE model reduce manual review work?
A: The **BGE model reduces manual review workload by almost two thirds** (from **45.7% to 16.7%** of tickets sent to review) compared to the TF-IDF + Logistic Regression model [model:confidence_rule][notebook:06#0][notebook:02b#5].
   mode: LLM (mistral: ministral-14b-latest) | sources: ['model:confidence_rule', 'notebook:06#0', 'notebook:02b#5', 'model:shap_bge', 'notebook:03b#3'] | numbers grounded: True
------------------------------------------------------------------------------------------------------------------------


Q: Are there recurring issues that span several categories?
A: Yes, the **BGE embeddings + Logistic Regression** model identifies recurring issues that span multiple categories, referred to as **cross-category topics**. Specifically:

- **Login-related issues** span **5 categories** (e.g., login/password failures merged into a broader "login" theme) with **3,460 tickets** [notebook:04b#7].
- **Billing-related issues** span multiple categories with **2,992 tickets** [notebook:04b#7].

The **TF-IDF + Logistic Regression** model does not explicitly highlight cross-category themes in the provided context. The BGE model’s ability to group by **shared meaning** (e.g., login failures, security incidents) enables this cross-category insight, unlike the word-based TF-IDF approach [notebook:04b#7].
   mode: LLM (mistral: ministral-14b-latest) | sources: ['themes:other', 'report#8', 'themes:security_concern', 'notebook:04b#6', 'notebook:04b#7'] | numbers grounded: True
---------------------------

Q: Can we trust the SHAP explanations of the embedding model?
A: Yes, the SHAP explanations for the **BGE sentence embeddings + Logistic Regression** model can be trusted based on the following evidence from the context documents:

- **Deletion Test (Faithfulness):** Deleting the top SHAP word lowers the confidence of the original prediction by **12 points** on average (vs **3 points** for a random word), demonstrating that the explanations point to words the model truly depends on. Deleting the top 3 SHAP words reduces confidence by **52 points** (vs **12 for random words**) and changes the prediction for **41% of messages** [notebook:03b#4].

- **Exact Shapley Values:** The method computes exact Shapley values by evaluating every possible subset of words (feasible due to short message lengths), ensuring no approximation errors. The additivity check shows an error of **4e-8**, matching the embedding model’s float precision [notebook:03b#0, notebook:03b#5].

- **Key Words in Explanatio

### Follow-up questions (chat)

In a chat, people ask short follow-ups like *"and which classes are still weak?"*. We pass the earlier turns as `history`.
The previous question is also used for retrieval, so the follow-up finds the right documents.

In [6]:
history = []
for q in ["How good is the BGE model?", "And which classes are still weak with it?"]:
    result = rag.answer(q, retriever, history=history)
    history += [{"role": "user", "content": q}, {"role": "assistant", "content": result["answer"]}]
    print("Q:", q)
    print("A:", result["answer"])
    print(f"   mode: {result['mode']} | sources: {result['sources']} | numbers grounded: {result['numbers_grounded']}")
    print("-" * 120)

Q: How good is the BGE model?
A: The **BGE sentence embeddings + Logistic Regression** model significantly outperforms the original **TF-IDF + Logistic Regression** model:

- **Macro F1**: 0.816 (vs 0.541 for TF-IDF) [model:summary].
- **Per-class improvements**:
  - **Account_access**: F1 jumps from 0.00 to 0.80.
  - **Other**: F1 improves from 0.00 to 0.49.
  - **Billing_problem**: F1 rises from 0.41 to 0.86.
  - **How_to**: F1 increases from 0.67 to 1.00.
  - **Security_concern**: F1 improves from 0.40 to 0.66 [notebook:02b#5].
- **Confidence rule efficiency**:
  - **Manual review share drops** from 45.7% (TF-IDF) to **16.7%** (BGE), reducing workload by **~66%** [model:confidence_rule].
  - **Auto-routing accuracy** improves from 92.4% (TF-IDF) to **95.0%** (BGE) [model:confidence_rule].
  - **Overall accuracy** rises from 58.5% (TF-IDF) to **83.3%** (BGE) [model:confidence_rule].

The BGE model also fixes **6 of 10 misclassified messages** from TF-IDF while introducing no new erro

Q: And which classes are still weak with it?
A: The **BGE sentence embeddings + Logistic Regression** model still struggles with certain classes, particularly those with **limited sentence variety** (the "minority" classes identified in [notebook:01#4]):

- **Account_access** and **other** remain the weakest, despite improvements:
  - **Account_access**: F1 rises from **0.00 (TF-IDF)** to **0.80 (BGE)**, but its sentences are still rare (only 3 unique examples) [notebook:02b#5].
  - **Other**: F1 improves from **0.00 (TF-IDF)** to **0.49 (BGE)**, but its vague phrasing (e.g., *"Something seems off…"*) overlaps with other categories [notebook:02b#5].
- **Security_concern** improves (F1: **0.40 → 0.66**), but still underperforms compared to others [notebook:02b#5].
- **Billing_problem** and **how_to** see strong gains (F1: **0.41 → 0.86** and **0.67 → 1.00**, respectively), but **feature_request** drops slightly (F1: **1.00 → 0.86**) due to misclassifying a security-related sentence [not

Without an API key, the answers come from the best-matching documents word for word, so they're always grounded but can read a bit like a report.
With a key (here Mistral), the LLM writes a shorter, conversational answer from the same documents and cites them in square brackets.

## 4. GenAI + SHAP: explain a single prediction

For a new ticket, we combine three things: the model's prediction, the SHAP words that drove it (notebook 03),
and the retrieved facts about that category. The assistant turns them into a short explanation for the support agent.
The dashboard does the same with the BGE model and its word-level SHAP (notebook 3b).

In [7]:
import joblib
import numpy as np

model = joblib.load(config.MODELS / "tfidf_logreg.joblib")
background = np.load(config.MODELS / "shap_background.npy")
tfidf = model.named_steps["tfidfvectorizer"]
clf = model.named_steps["logisticregression"]

def predict_and_explain(text):
    proba = model.predict_proba([text])[0]
    i = int(np.argmax(proba))
    x = tfidf.transform([text]).toarray()[0]
    contrib = (x - background) * clf.coef_[i]                     # SHAP values for the predicted class
    present = np.where(x > 0)[0]
    terms = [(tfidf.get_feature_names_out()[j], contrib[j]) for j in present[np.argsort(-np.abs(contrib[present]))]]
    explanation, mode = rag.explain_prediction(text, clf.classes_[i], proba[i], terms, retriever,
                                              model_name="TF-IDF + Logistic Regression")
    return clf.classes_[i], proba[i], explanation, mode

for text in ["I was charged twice for my subscription this month.",
             "How do I reset my password?"]:
    label, conf, explanation, mode = predict_and_explain(text)
    print(f"Ticket: {text}\nPrediction: {label} ({conf:.0%})\nExplanation ({mode}): {explanation}\n")

Ticket: I was charged twice for my subscription this month.
Prediction: billing_problem (99%)
Explanation (LLM): This ticket was classified as a **billing_problem** with **99% confidence** by the **BGE sentence embeddings + Logistic Regression** model (the best model) [theme:billing_problem#0].

The prediction is strong because the description contains highly relevant keywords like **"charged"** and **"charged twice"**, which directly align with billing-related issues (e.g., duplicate charges). The model also associates **"month"** and **"my subscription"** with billing disputes, reinforcing the classification. No conflicting terms were detected, further solidifying the prediction [theme:billing_problem#0].



Ticket: How do I reset my password?
Prediction: account_access (40%)
Explanation (LLM): For the **TF-IDF + Logistic Regression model**, the prediction leaned toward *account_access* because the ticket contained the high-impact phrases **"my password"** and **"password"**, which align with the model’s learned association between password-related terms and account access issues (e.g., "account was," "failed login") [shap:account_access].

However, the **BGE embedding + Logistic Regression model** would likely have been more confident in this prediction, as **"password"** (SHAP value: **0.90**) is its strongest indicator for *account_access*, while the word **"how"** (a functional query) weakly pushed against the classification [shap:account_access].

The low confidence (40%) reflects the model’s uncertainty due to the lack of explicit *account access* keywords (e.g., "failed," "locked," or "2fa") and the presence of a generic question structure.



## 5. The grounding check

Every number in an answer must appear in the documents the assistant was given. Here's what happens with an honest answer and with a made-up one:

In [8]:
context = kb.loc[kb["doc_id"] == "analytics:satisfaction", "text"].iloc[0]
honest = "Performance tickets are the most negative at 60.8%, with a CSAT of 2.80."
invented = "Performance tickets are the most negative at 75%, and complaints doubled last month."
print("Honest answer grounded?  ", rag.numbers_grounded(honest, context))
print("Invented answer grounded?", rag.numbers_grounded(invented, context))

Honest answer grounded?   True
Invented answer grounded? False


The invented "75%" isn't in the source, so the answer is flagged. The dashboard shows a warning next to any answer that fails this check.

## 6. Save the knowledge base

In [9]:
kb.to_csv(config.PROCESSED / "knowledge_base.csv", index=False)
retrieval.to_csv(config.PROCESSED / "rag_retrieval_eval.csv", index=False)
print("Saved knowledge_base.csv:", kb.shape, "| rag_retrieval_eval.csv")

Saved knowledge_base.csv: (147, 4) | rag_retrieval_eval.csv


## Summary

| Part | What we did | Result |
|---|---|---|
| Knowledge base | ~150 documents: data documents from every saved result, plus the written interpretation of every notebook section, the evaluation report and the README | Updates automatically when results change |
| Retrieval | Hybrid: TF-IDF keywords + BGE semantic search | Right document first for 79% of 28 test questions, in the top 5 for 96% |
| Generation | Mistral (from `.env`) with automatic fallback between models; Claude / OpenAI-compatible also supported; offline mode | Always answers, even without internet |
| Chat | Follow-up questions use the earlier turns | Short follow-ups find the right documents |
| Explainability | SHAP words + retrieved category facts | Plain-English reason for each prediction |
| Safety | Masking, context-only prompt, number check | Made-up numbers are caught |

**To use an LLM:** put the Mistral key in `.env` (either `MISTRAL_API_KEY=...` or just the key on one line), or set `ANTHROPIC_API_KEY` / `OPENAI_API_KEY`.
Model names are in `src/config.py`.